<a href="https://colab.research.google.com/github/Suim00/hdac-knhanes-skipbreakfast-obesity/blob/main/code_05_table1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5주차 팀 실습 — 공복혈당 ≥ 100 mg/dL 층화 Table 1 (B-2)
- 노출군 = `glucose >= 100` / 비노출군 = `glucose < 100` / 결측(6.8%)은 어느 군에도 넣지 않음
- **노출군·비노출군 열은 자료를 자르지 않고 도메인 분석으로 산출** (층·조사구·가중치 유지)
- 열: 전체 / 노출군 / 비노출군 + SMD(노출 vs 비노출)
- 실행 순서: 0 → 1 → 2 → 3 → 4 (`knhanes_like_week4.csv`가 같은 폴더에 있어야 함)

## 0. 자료 불러오기

In [3]:
import numpy as np, pandas as pd
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

# Colab 이면 CSV 업로드 (로컬이면 그냥 통과)
try:
    import os
    from google.colab import files
    if not os.path.exists('knhanes_like_week4.csv'):
        files.upload()
except ImportError:
    pass

df = pd.read_csv('knhanes_like_week4.csv')
w  = df['wt'].to_numpy()          # 가중치
ST = df['kstrata'].to_numpy()     # 층
PS = df['psu'].to_numpy()         # 집락(조사구)
print(f'전체 n = {len(df):,} | 조사구 {df.psu.nunique()} | 층 {df.kstrata.nunique()}')

전체 n = 5,367 | 조사구 263 | 층 18


## 1. 복합표본 함수 (4주차와 동일)
`svymean_domain`은 도메인 밖 관측치의 가중치만 0으로 만들 뿐, 층·조사구 구조에서 빼지 않는다 → 도메인 분석.

In [4]:
def _var_psu(z, strata, psu):
    t = pd.DataFrame({'st': strata, 'cl': psu, 'z': z})
    zc = t.groupby(['st', 'cl'])['z'].sum().reset_index()
    var = 0.0
    for s, g in zc.groupby('st'):
        m = len(g)
        if m > 1:
            var += m / (m - 1) * np.sum((g['z'] - g['z'].mean()) ** 2)
    return var

def svymean_domain(y, w, strata, psu, domain):
    """도메인 평균/비율과 설계 기반 SE (y 의 NaN 은 자동 제외)."""
    y = np.asarray(y, float)
    d = np.asarray(domain, float) * (~np.isnan(y))
    y = np.nan_to_num(y) * d
    wd = np.asarray(w, float) * d
    th = np.sum(wd * y) / np.sum(wd)
    se = np.sqrt(_var_psu(wd * (y - th) / np.sum(wd), strata, psu))
    return th, se

def wsd(y, w):
    """가중 표준편차 (SMD 분모용; SE 가 아니므로 잘라서 계산해도 무방)."""
    y = np.asarray(y, float); w = np.asarray(w, float)
    ok = ~np.isnan(y); y, w = y[ok], w[ok]
    m = np.sum(w * y) / np.sum(w)
    return np.sqrt(np.sum(w * (y - m) ** 2) / np.sum(w))

## 2. 노출 정의와 결측 확인
`NaN >= 100` 은 False 이므로 결측자가 비노출군으로 **잘못 들어가기 쉽다** → `glu_obs`(관측 여부)를 명시적으로 곱한다.

In [5]:
glu_obs = df['glucose'].notna().to_numpy()
EXP     = glu_obs & (df['glucose'] >= 100).to_numpy()   # 노출군
UNEXP   = glu_obs & (df['glucose'] <  100).to_numpy()   # 비노출군
assert (EXP & UNEXP).sum() == 0 and EXP.sum() + UNEXP.sum() == glu_obs.sum()

print(f'공복혈당 결측 : {(~glu_obs).sum():,}명 ({(~glu_obs).mean()*100:.1f}%)')
print(f'노출군  (≥100): {EXP.sum():,}명')
print(f'비노출군(<100): {UNEXP.sum():,}명')
prev, se = svymean_domain(EXP.astype(float), w, ST, PS, glu_obs)
print(f'가중 유병률    : {prev*100:.1f}% (95% CI {100*(prev-1.96*se):.1f}-{100*(prev+1.96*se):.1f})')

# 결측군 vs 관측군 (가중, 도메인)
miss = ~glu_obs
chk = []
for lab, y in [('연령', df['age']), ('고혈압', df['htn'].astype(float))]:
    a, _ = svymean_domain(y, w, ST, PS, glu_obs)
    b, _ = svymean_domain(y, w, ST, PS, miss)
    chk.append([lab, round(a, 2), round(b, 2)])
print(pd.DataFrame(chk, columns=['변수', '혈당 관측군(가중)', '혈당 결측군(가중)']).to_string(index=False))

공복혈당 결측 : 367명 (6.8%)
노출군  (≥100): 1,721명
비노출군(<100): 3,279명
가중 유병률    : 31.7% (95% CI 30.4-33.1)
 변수  혈당 관측군(가중)  혈당 결측군(가중)
 연령       48.96       51.75
고혈압        0.24        0.30


## 3. Table 1 — 전체 / 노출군 / 비노출군 (도메인 분석)
- 범주형 `n (가중 %)`, 연속형 `가중 평균 (SE)`
- SMD = (노출군 − 비노출군) ÷ 합동 SD, **`*` = 반올림 전 |SMD| ≥ 0.10 (0.10 이상)**

In [6]:
DOMS = {'전체': glu_obs, '노출군': EXP, '비노출군': UNEXP}

def ind(cond, src):
    """지시변수. 원변수가 결측이면 NaN 으로 남긴다."""
    return np.where(df[src].isna(), np.nan, cond.astype(float))

ROWS = [
    ('연령 (세)',            'cont', df['age'].astype(float).to_numpy()),
    ('65세 이상',            'cat',  ind(df.age >= 65, 'age')),
    ('남성',                 'cat',  ind(df.sex == 1, 'sex')),
    ('읍면 거주',            'cat',  ind(df.urban == 2, 'urban')),
    ('BMI (kg/m²)',          'cont', df['bmi'].astype(float).to_numpy()),
    ('비만 (BMI≥25)',        'cat',  ind(df.bmi >= 25, 'bmi')),
    ('수축기혈압 (mmHg)',    'cont', df['sbp'].astype(float).to_numpy()),
    ('고혈압',               'cat',  df['htn'].astype(float).to_numpy()),
    ('총콜레스테롤 (mg/dL)', 'cont', df['chol'].astype(float).to_numpy()),
    ('현재흡연',             'cat',  ind(df.smk == 1, 'smk')),
    ('고위험음주',           'cat',  ind(df.drink == 3, 'drink')),
    ('규칙적 운동',          'cat',  ind(df.exercise == 2, 'exercise')),
]

def smd(y, kind, g1, g2):
    y = np.asarray(y, float)
    m1, _ = svymean_domain(y, w, ST, PS, g1)
    m2, _ = svymean_domain(y, w, ST, PS, g2)
    if kind == 'cat':
        s = np.sqrt((m1 * (1 - m1) + m2 * (1 - m2)) / 2)
    else:
        s = np.sqrt((wsd(y[g1], w[g1]) ** 2 + wsd(y[g2], w[g2]) ** 2) / 2)
    return (m1 - m2) / s

cols = {'전체': f'전체 (n={glu_obs.sum():,})',
        '노출군': f'노출군 ≥100 (n={EXP.sum():,})',
        '비노출군': f'비노출군 <100 (n={UNEXP.sum():,})'}

out, smd_raw = [], {}
for lab, kind, y in ROWS:
    r = {'특성': lab}
    for k, dom in DOMS.items():
        th, se = svymean_domain(y, w, ST, PS, dom)
        n_obs = int((dom & ~np.isnan(y)).sum())
        if kind == 'cont':
            s = f'{th:.1f} ({se:.2f})'
            if n_obs < dom.sum():                      # 결측 있는 변수는 n 병기
                s += f' [n={n_obs:,}]'
        else:
            s = f'{int(np.nansum(y * dom)):,} ({th * 100:.1f})'
        r[cols[k]] = s
    v = smd(y, kind, EXP, UNEXP)
    smd_raw[lab] = v
    if round(v, 2) == 0:
        r['SMD'] = '0.00'
    else:
        r['SMD'] = f'{v:+.2f}' + (' *' if abs(v) >= 0.10 else '')   # 반올림 전 기준, 0.10 이상
    out.append(r)

table1 = pd.DataFrame(out)
print(table1.to_string(index=False))
table1.to_csv('table1_glucose100.csv', index=False, encoding='utf-8-sig')
print()
print('SMD 반올림 전 값 (별표 기준 확인):')
print(pd.Series(smd_raw).round(4).to_string())

            특성           전체 (n=5,000)     노출군 ≥100 (n=1,721)    비노출군 <100 (n=3,279)     SMD
        연령 (세)            49.0 (0.18)            53.3 (0.41)            47.0 (0.21) +0.40 *
        65세 이상           1,466 (16.4)             699 (24.6)             767 (12.5) +0.31 *
            남성           2,543 (49.2)             824 (46.4)           1,719 (50.5)   -0.08
         읍면 거주           1,985 (22.6)             751 (25.4)           1,234 (21.2)   +0.10
   BMI (kg/m²)            23.8 (0.06)            24.8 (0.10)            23.3 (0.06) +0.40 *
   비만 (BMI≥25)           1,890 (36.5)             838 (47.8)           1,052 (31.3) +0.34 *
  수축기혈압 (mmHg)           122.6 (0.45)           126.6 (0.56)           120.7 (0.51) +0.33 *
           고혈압           1,517 (24.0)             655 (30.8)             862 (20.9) +0.23 *
총콜레스테롤 (mg/dL) 170.2 (0.50) [n=4,484] 172.8 (0.84) [n=1,547] 169.0 (0.60) [n=2,937] +0.12 *
          현재흡연           1,206 (24.2)             416 (24.1)             790 (24

## 4. 보고서에 붙여넣을 마크다운 표 + 각주

In [7]:
def to_md(t):
    head = '| ' + ' | '.join(t.columns) + ' |'
    sep  = '| :--- | :---: | :---: | :---: | :---: |'
    lines = [head, sep]
    for _, r in t.iterrows():
        cells = [f'**{r.iloc[0]}**'] + [str(c).replace(' [n=', '<br>[n=') for c in r.iloc[1:]]
        lines.append('| ' + ' | '.join(cells) + ' |')
    return '\n'.join(lines)

print(to_md(table1))
print('''
[각주]
1. 모든 수치(빈도 제외)는 층, 집락(조사구), 가중치를 반영한 복합표본설계 추정치이며, n은 결측치(약 6.8%)를 제외한 유효 관측 수입니다.
2. 연속형 변수는 가중 평균과 설계 기반 표준오차로, 범주형 변수는 가중 백분율로 제시하였습니다.
3. 노출군(공복혈당 ≥ 100 mg/dL)과 비노출군 간의 층화 분석은 표본 임의의 절단으로 인한 표준오차 왜곡을 방지하고자 도메인 분석 기법을 적용하였습니다. SMD는 두 군의 가중 추정치 차이를 합동 SD로 나눈 값이며, *는 반올림 전 |SMD|가 0.10 이상(0.10 포함)인 경우입니다.
4. 공복혈당 검사 결과가 누락된 결측 데이터(6.8%)는 완전 사례 분석(Complete-case analysis) 원칙에 따라 분모 계산에서 배제하였습니다. 총콜레스테롤(결측 10.5%)은 관측된 값만 사용하여 [n]을 병기하였습니다.''')

| 특성 | 전체 (n=5,000) | 노출군 ≥100 (n=1,721) | 비노출군 <100 (n=3,279) | SMD |
| :--- | :---: | :---: | :---: | :---: |
| **연령 (세)** | 49.0 (0.18) | 53.3 (0.41) | 47.0 (0.21) | +0.40 * |
| **65세 이상** | 1,466 (16.4) | 699 (24.6) | 767 (12.5) | +0.31 * |
| **남성** | 2,543 (49.2) | 824 (46.4) | 1,719 (50.5) | -0.08 |
| **읍면 거주** | 1,985 (22.6) | 751 (25.4) | 1,234 (21.2) | +0.10 |
| **BMI (kg/m²)** | 23.8 (0.06) | 24.8 (0.10) | 23.3 (0.06) | +0.40 * |
| **비만 (BMI≥25)** | 1,890 (36.5) | 838 (47.8) | 1,052 (31.3) | +0.34 * |
| **수축기혈압 (mmHg)** | 122.6 (0.45) | 126.6 (0.56) | 120.7 (0.51) | +0.33 * |
| **고혈압** | 1,517 (24.0) | 655 (30.8) | 862 (20.9) | +0.23 * |
| **총콜레스테롤 (mg/dL)** | 170.2 (0.50)<br>[n=4,484] | 172.8 (0.84)<br>[n=1,547] | 169.0 (0.60)<br>[n=2,937] | +0.12 * |
| **현재흡연** | 1,206 (24.2) | 416 (24.1) | 790 (24.2) | 0.00 |
| **고위험음주** | 1,230 (24.9) | 432 (25.9) | 798 (24.5) | +0.03 |
| **규칙적 운동** | 2,135 (42.5) | 747 (43.4) | 1,388 (42.1) | +0.03 |

[각주]
1. 모든 수치(빈도 제외)는 층, 집락(조사구), 가중

## 5. (B-3용) 비가중 vs 가중 — 분석 대상 5,000명 기준
순위 기준 = 표준화 차이 = (가중 − 비가중) ÷ 가중 SD (단위가 다른 변수끼리 비교 가능)

In [8]:
rec = []
for lab, kind, y in ROWS:
    y = np.asarray(y, float); sc = 100 if kind == 'cat' else 1
    d = glu_obs & ~np.isnan(y)                       # 5,000명 기준 (결측 제외)
    unw = y[d].mean()
    th, _ = svymean_domain(y, w, ST, PS, glu_obs)
    rec.append({'변수': lab, '비가중': round(unw * sc, 1), '가중': round(th * sc, 1),
                '차이': round((th - unw) * sc, 1),
                '표준화 차이': round((th - unw) / wsd(y[d], w[d]), 2)})
cmp = pd.DataFrame(rec)
cmp['순위'] = cmp['표준화 차이'].abs().rank(ascending=False).astype(int)
print(cmp.sort_values('순위')[['순위', '변수', '비가중', '가중', '차이', '표준화 차이']].to_string(index=False))

 순위             변수   비가중    가중    차이  표준화 차이
  1          읍면 거주  39.7  22.6 -17.1   -0.41
  2         65세 이상  29.3  16.4 -13.0   -0.35
  3         연령 (세)  53.7  49.0  -4.8   -0.30
  4   수축기혈압 (mmHg) 125.4 122.6  -2.8   -0.16
  5            고혈압  30.3  24.0  -6.3   -0.15
  6 총콜레스테롤 (mg/dL) 171.9 170.2  -1.7   -0.05
  7    BMI (kg/m²)  23.9  23.8  -0.1   -0.04
  8             남성  50.9  49.2  -1.7   -0.03
  8    비만 (BMI≥25)  37.8  36.5  -1.3   -0.03
 10          고위험음주  24.6  24.9   0.3    0.01
 11           현재흡연  24.1  24.2   0.0    0.00
 11         규칙적 운동  42.7  42.5  -0.2   -0.00
